# Import Packages & Data, Prepare Data

In [4]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold

In [20]:
# Load the datasets
interactions = pd.read_csv('https://raw.githubusercontent.com/sandrotissi/book-recommender/0ca2f3f0ea5d72dd53b107ed15ea40401a48e74b/interactions_train.csv')

In [6]:
# Let's first sort the interactions by user and time stamp
interactions = interactions.sort_values(["u", "t"])

In [7]:
interactions["pct_rank"] = interactions.groupby("u")["t"].rank(pct=True, method='dense')
interactions.reset_index(inplace=True, drop=True)

In [8]:
# Define a function to create the data matrix
def create_data_matrix(data, n_users, n_items):
    data_matrix = np.zeros((n_users, n_items))
    data_matrix[data["u"].values, data["i"].values] = 1
    return data_matrix

In [9]:
# Create the user-item matrix using the full 'interactions' DataFrame
# Ensure n_users and n_items are based on the full dataset's max values
full_n_users = interactions['u'].max() + 1
full_n_items = interactions['i'].max() + 1

full_data_matrix = create_data_matrix(interactions, full_n_users, full_n_items)

In [10]:
# Compute the item-item similarity matrix using the full data matrix
full_item_similarity = cosine_similarity(full_data_matrix.T)

In [11]:
# Define the function to predict interactions based on item similarity
def item_based_predict(interactions, similarity, epsilon=1e-9):
    # np.dot does the matrix multiplication. Here we are calculating the
    # weighted sum of interactions based on item similarity
    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon)
    return pred.T  # Transpose to get users as rows and items as columns

# Calculate the item-based predictions for positive interactions using full data
full_item_prediction = item_based_predict(full_data_matrix, full_item_similarity)

In [12]:
# Define the precision_recall_at_k function here to ensure it's available for subsequent evaluation steps
def precision_recall_at_k(prediction, ground_truth, k=10):
    num_users = prediction.shape[0]
    precision_at_k, recall_at_k = 0, 0

    for user in range(num_users):
        # Get the indices of the top-K items for the user based on predicted scores
        top_k_items = np.argsort(prediction[user, :])[-k:]

        # Calculate the number of relevant items in the top-K items for the user
        relevant_items_in_top_k = np.isin(top_k_items, np.where(ground_truth[user, :] == 1)[0]).sum()

        # Calculate the total number of relevant items for the user
        total_relevant_items = ground_truth[user, :].sum()

        # Precision@K and Recall@K for this user
        precision_at_k += relevant_items_in_top_k / k
        recall_at_k += relevant_items_in_top_k / total_relevant_items if total_relevant_items > 0 else 0

    # Average Precision@K and Recall@K over all users
    precision_at_k /= num_users
    recall_at_k /= num_users

    return precision_at_k, recall_at_k

In [13]:
# Load rereading probabilities
reread_probs = pd.read_csv('https://raw.githubusercontent.com/sandrotissi/book-recommender/54aae1badcbff4d4cb8f1fe8bf22827aa4f0ea50/book_rereading_probabilities.csv')


# Filter for items with probability > 25%
# Based on inspection, the column name is 'reread_probability' and 'book_id'
high_prob_books = reread_probs[reread_probs['reread_probability'] > 0.25]['book_id'].values

# Create the filtered already-read matrix
# We take the full interaction matrix and zero out books that DON'T have a high reread probability
filtered_already_read_matrix = full_data_matrix.copy()

# Get all book IDs (columns) and find those NOT in our high-probability list
all_book_ids = np.arange(full_data_matrix.shape[1])
low_prob_mask = ~np.isin(all_book_ids, high_prob_books)

# Set interactions for low-probability books to 0
filtered_already_read_matrix[:, low_prob_mask] = 0

print(f"Books with >25% reread probability: {len(high_prob_books)}")
print(f"Total filtered interactions kept: {np.sum(filtered_already_read_matrix)}")

Books with >25% reread probability: 5103
Total filtered interactions kept: 14468.0


In [14]:
# Define the functions to ensure they are available
def user_based_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions) / (np.abs(similarity).sum(axis=1)[:, np.newaxis] + epsilon)
    return pred

# Create Weighted Predictions

In [21]:
# Ensure full_user_similarity is calculated
full_user_similarity = cosine_similarity(full_data_matrix)

# Ensure full_user_prediction is calculated
full_user_prediction = user_based_predict(full_data_matrix, full_user_similarity)

# Define current_n_items (was previously in a deleted cell)
current_n_items = interactions['i'].max() + 1

# Initialize a list to store recommendations for all users
all_next_10_i_recommendations = []

# Get the maximum item ID available in the dataset
max_available_item_id = interactions['i'].max()

# Group interactions by user to find the highest 'i' for each user
user_max_i = interactions.groupby('u')['i'].max()

# Loop through each user to generate recommendations
for u_id in range(full_n_users):
    # Get the highest 'i' the current user has interacted with
    # If a user has no interactions, we'll default to -1 or handle as needed.
    # For this task, we assume all users in n_users have at least one interaction or default max_i from user_max_i
    current_user_max_i = user_max_i.get(u_id, -1) # Default to -1 if user has no interactions

    # Generate the next 3 consecutive item IDs
    recommended_items = []
    for j in range(1, 3):
        next_i = current_user_max_i + j
        # Ensure the recommended item ID does not exceed the maximum available item ID
        if next_i <= max_available_item_id:
            recommended_items.append(next_i)
        else:
            # If we run out of valid item IDs, stop adding to this user's recommendations
            break
    all_next_10_i_recommendations.append({'user_id': u_id, 'recommendation_next_10_i': recommended_items})

# Create a DataFrame from the collected recommendations
next_10_i_recommendations_df = pd.DataFrame(all_next_10_i_recommendations)

# Create a score matrix for the 'next 3 books' recommendations
# This matrix will be of the same shape as user_prediction, with 1s for recommended items and 0s otherwise.
next_10_reco_matrix = np.zeros((full_n_users, current_n_items))

for index, row in next_10_i_recommendations_df.iterrows():
    user_id = row['user_id']
    recommended_items = row['recommendation_next_10_i']
    for item_id in recommended_items:
        if item_id < current_n_items: # Ensure item_id is within bounds
            next_10_reco_matrix[user_id, item_id] = 1

# Update weights and re-calculate the combined prediction
w_user_cf = 0.06
w_item_cf = 0.06
w_next_10 = 0.06
w_already_read = 0.82

# Standardizing variables
num_users_with_next_10 = 3400
weighted_combined_filtered = np.zeros_like(full_user_prediction)

for u_id in range(full_n_users):
    base_cf = (w_user_cf * full_user_prediction[u_id, :]) + (w_item_cf * full_item_prediction[u_id, :])
    read_comp = (w_already_read * filtered_already_read_matrix[u_id, :])

    if u_id < num_users_with_next_10:
        weighted_combined_filtered[u_id, :] = base_cf + read_comp + (w_next_10 * next_10_reco_matrix[u_id, :])
    else:
        adj_w_user = w_user_cf
        adj_w_item = w_item_cf
        weighted_combined_filtered[u_id, :] = (adj_w_user * full_user_prediction[u_id, :]) + (adj_w_item * full_item_prediction[u_id, :]) + read_comp

In [22]:
all_prob_weighted_recos = []

for u_id in range(full_n_users):
    # Get the predicted scores for the current user
    user_scores = weighted_combined_filtered[u_id, :].copy()

    # Get top 10 item indices
    top_k_item_ids = np.argsort(user_scores)[-10:][::-1]
    reco_str = ' '.join(map(str, top_k_item_ids))
    all_prob_weighted_recos.append({'user_id': u_id, 'recommendation': reco_str})

prob_weighted_df = pd.DataFrame(all_prob_weighted_recos)
prob_weighted_df.to_csv('weighted_model.csv', index=False)

# Evaluate User-User Collaborative Filtering using K-Fold Cross-Validation

In [17]:
# Define the number of folds for cross-validation
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Lists to store precision and recall values for each fold
all_precision_at_k = []
all_recall_at_k = []

k_value_cv = 10  # K for evaluation in cross-validation

print(f"Starting K-Fold Cross-Validation with {n_splits} splits and K={k_value_cv}...")

for fold, (train_index, test_index) in enumerate(kf.split(interactions)):
    print(f"\n--- Fold {fold + 1}/{n_splits} ---")

    # Split data into training and testing sets for the current fold
    train_data_fold = interactions.iloc[train_index]
    test_data_fold = interactions.iloc[test_index]

    # Create data matrices for the current fold
    # Use full_n_users and full_n_items to maintain consistent dimensions across all folds
    train_data_matrix_fold = create_data_matrix(train_data_fold, full_n_users, full_n_items)
    test_data_matrix_fold = create_data_matrix(test_data_fold, full_n_users, full_n_items)

    # Calculate user-user similarity based on the training data of the current fold
    user_similarity_train_fold = cosine_similarity(train_data_matrix_fold)

    # Make predictions
    user_prediction_train_fold = user_based_predict(train_data_matrix_fold, user_similarity_train_fold)

    # Evaluate Precision@K and Recall@K
    precision_fold, recall_fold = precision_recall_at_k(user_prediction_train_fold, test_data_matrix_fold, k=k_value_cv)

    all_precision_at_k.append(precision_fold)
    all_recall_at_k.append(recall_fold)

    print(f"Precision@{k_value_cv} for Fold {fold + 1}: {precision_fold:.4f}")
    print(f"Recall@{k_value_cv} for Fold {fold + 1}: {recall_fold:.4f}")

# Calculate and print the average precision and recall across all folds
mean_precision = np.mean(all_precision_at_k)
mean_recall = np.mean(all_recall_at_k)

print(f"\n--- Cross-Validation Results (Average over {n_splits} folds) ---")
print(f"Average User-User CF Precision@{k_value_cv}: {mean_precision:.4f}")
print(f"Average User-User CF Recall@{k_value_cv}: {mean_recall:.4f}")

Starting K-Fold Cross-Validation with 5 splits and K=10...

--- Fold 1/5 ---
Precision@10 for Fold 1: 0.0477
Recall@10 for Fold 1: 0.2187

--- Fold 2/5 ---
Precision@10 for Fold 2: 0.0470
Recall@10 for Fold 2: 0.2199

--- Fold 3/5 ---
Precision@10 for Fold 3: 0.0483
Recall@10 for Fold 3: 0.2246

--- Fold 4/5 ---
Precision@10 for Fold 4: 0.0471
Recall@10 for Fold 4: 0.2234

--- Fold 5/5 ---
Precision@10 for Fold 5: 0.0478
Recall@10 for Fold 5: 0.2200

--- Cross-Validation Results (Average over 5 folds) ---
Average User-User CF Precision@10: 0.0476
Average User-User CF Recall@10: 0.2213


# Evaluate Item-Item Collaborative Filtering using K-Fold Cross-Validation

In [18]:
# Define the number of folds for cross-validation
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Lists to store precision and recall values for each fold
all_precision_at_k_item_cf = []
all_recall_at_k_item_cf = []

k_value_cv = 10  # K for evaluation in cross-validation

print(f"Starting K-Fold Cross-Validation for Item-Item CF with {n_splits} splits and K={k_value_cv}...")

for fold, (train_index, test_index) in enumerate(kf.split(interactions)):
    print(f"\n--- Fold {fold + 1}/{n_splits} ---")

    # Split data into training and testing sets for the current fold
    train_data_fold = interactions.iloc[train_index]
    test_data_fold = interactions.iloc[test_index]

    # Create data matrices for the current fold
    # Use full_n_users and full_n_items to maintain consistent dimensions across all folds
    train_data_matrix_fold = create_data_matrix(train_data_fold, full_n_users, full_n_items)
    test_data_matrix_fold = create_data_matrix(test_data_fold, full_n_users, full_n_items)

    # Calculate item-item similarity based on the training data of the current fold
    # Note the .T (transpose) to get items as rows for similarity calculation
    item_similarity_train_fold = cosine_similarity(train_data_matrix_fold.T)

    # Make predictions using the item-based collaborative filtering approach
    item_prediction_train_fold = item_based_predict(train_data_matrix_fold, item_similarity_train_fold)

    # Evaluate Precision@K and Recall@K
    precision_fold_item_cf, recall_fold_item_cf = precision_recall_at_k(item_prediction_train_fold, test_data_matrix_fold, k=k_value_cv)

    all_precision_at_k_item_cf.append(precision_fold_item_cf)
    all_recall_at_k_item_cf.append(recall_fold_item_cf)

    print(f"Item-Item CF Precision@{k_value_cv} for Fold {fold + 1}: {precision_fold_item_cf:.4f}")
    print(f"Item-Item CF Recall@{k_value_cv} for Fold {fold + 1}: {recall_fold_item_cf:.4f}")

# Calculate and print the average precision and recall across all folds
mean_precision_item_cf = np.mean(all_precision_at_k_item_cf)
mean_recall_item_cf = np.mean(all_recall_at_k_item_cf)

print(f"\n--- Item-Item CF Cross-Validation Results (Average over {n_splits} folds) ---")
print(f"Average Item-Item CF Precision@{k_value_cv}: {mean_precision_item_cf:.4f}")
print(f"Average Item-Item CF Recall@{k_value_cv}: {mean_recall_item_cf:.4f}")

Starting K-Fold Cross-Validation for Item-Item CF with 5 splits and K=10...

--- Fold 1/5 ---
Item-Item CF Precision@10 for Fold 1: 0.0461
Item-Item CF Recall@10 for Fold 1: 0.1981

--- Fold 2/5 ---
Item-Item CF Precision@10 for Fold 2: 0.0462
Item-Item CF Recall@10 for Fold 2: 0.1991

--- Fold 3/5 ---
Item-Item CF Precision@10 for Fold 3: 0.0467
Item-Item CF Recall@10 for Fold 3: 0.2036

--- Fold 4/5 ---
Item-Item CF Precision@10 for Fold 4: 0.0461
Item-Item CF Recall@10 for Fold 4: 0.2015

--- Fold 5/5 ---
Item-Item CF Precision@10 for Fold 5: 0.0467
Item-Item CF Recall@10 for Fold 5: 0.1987

--- Item-Item CF Cross-Validation Results (Average over 5 folds) ---
Average Item-Item CF Precision@10: 0.0464
Average Item-Item CF Recall@10: 0.2002


# Evaluate Weighted Collaborative Filtering using K-Fold Cross-Validation

In [19]:
# Define the number of folds for cross-validation
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Lists to store precision and recall values for each fold
all_precision_at_k_weighted_cf = []
all_recall_at_k_weighted_cf = []

k_value_cv = 10  # K for evaluation in cross-validation

# Define the weights
w_user_cf = 0.06
w_item_cf = 0.06
w_next_10 = 0.06
w_already_read = 0.82
num_users_with_next_10 = 3400

print(f"Starting K-Fold Cross-Validation for Weighted CF with {n_splits} splits and K={k_value_cv}...")

for fold, (train_index, test_index) in enumerate(kf.split(interactions)):
    print(f"\n--- Fold {fold + 1}/{n_splits} ---")

    # Split data into training and testing sets for the current fold
    train_data_fold = interactions.iloc[train_index]
    test_data_fold = interactions.iloc[test_index]

    # Create data matrices for the current fold, maintaining full dimensions
    train_data_matrix_fold = create_data_matrix(train_data_fold, full_n_users, full_n_items)
    test_data_matrix_fold = create_data_matrix(test_data_fold, full_n_users, full_n_items)

    # 1. User-User CF prediction for the fold
    user_similarity_train_fold = cosine_similarity(train_data_matrix_fold)
    user_prediction_train_fold = user_based_predict(train_data_matrix_fold, user_similarity_train_fold)

    # 2. Item-Item CF prediction for the fold
    item_similarity_train_fold = cosine_similarity(train_data_matrix_fold.T)
    item_prediction_train_fold = item_based_predict(train_data_matrix_fold, item_similarity_train_fold)

    # 3. Filtered Already Read Matrix for the fold
    filtered_already_read_matrix_fold = train_data_matrix_fold.copy()
    # Apply the global low_prob_mask (derived from high_prob_books from full dataset)
    filtered_already_read_matrix_fold[:, low_prob_mask] = 0

    # 4. Next 2 Recommendations Matrix for the fold
    next_10_reco_matrix_fold = np.zeros((full_n_users, full_n_items))
    user_max_i_fold = train_data_fold.groupby('u')['i'].max()

    for u_id_fold in range(full_n_users):
        current_user_max_i_fold = user_max_i_fold.get(u_id_fold, -1)
        for j in range(1, 3): # As per original logic, next 2 items
            next_i_candidate = current_user_max_i_fold + j
            if next_i_candidate < full_n_items: # Ensure item ID is within bounds
                next_10_reco_matrix_fold[u_id_fold, next_i_candidate] = 1

    # Combine predictions with the specified weights
    weighted_combined_prediction_fold = np.zeros_like(user_prediction_train_fold)

    for u_id_for_weighting in range(full_n_users):
        base_cf_fold = (w_user_cf * user_prediction_train_fold[u_id_for_weighting, :]) + \
                       (w_item_cf * item_prediction_train_fold[u_id_for_weighting, :])
        read_comp_fold = (w_already_read * filtered_already_read_matrix_fold[u_id_for_weighting, :])

        if u_id_for_weighting < num_users_with_next_10:
            weighted_combined_prediction_fold[u_id_for_weighting, :] = base_cf_fold + read_comp_fold + \
                                                                       (w_next_10 * next_10_reco_matrix_fold[u_id_for_weighting, :])
        else:
            # For users beyond num_users_with_next_10, the next_10 component is not added
            weighted_combined_prediction_fold[u_id_for_weighting, :] = base_cf_fold + read_comp_fold

    # Evaluate the combined prediction
    precision_fold_weighted, recall_fold_weighted = precision_recall_at_k(weighted_combined_prediction_fold, test_data_matrix_fold, k=k_value_cv)

    all_precision_at_k_weighted_cf.append(precision_fold_weighted)
    all_recall_at_k_weighted_cf.append(recall_fold_weighted)

    print(f"Weighted CF Precision@{k_value_cv} for Fold {fold + 1}: {precision_fold_weighted:.4f}")
    print(f"Weighted CF Recall@{k_value_cv} for Fold {fold + 1}: {recall_fold_weighted:.4f}")

# Calculate and print the average precision and recall across all folds
mean_precision_weighted_cf = np.mean(all_precision_at_k_weighted_cf)
mean_recall_weighted_cf = np.mean(all_recall_at_k_weighted_cf)

print(f"\n--- Weighted CF Cross-Validation Results (Average over {n_splits} folds) ---")
print(f"Average Weighted CF Precision@{k_value_cv}: {mean_precision_weighted_cf:.4f}")
print(f"Average Weighted CF Recall@{k_value_cv}: {mean_recall_weighted_cf:.4f}")

Starting K-Fold Cross-Validation for Weighted CF with 5 splits and K=10...

--- Fold 1/5 ---
Weighted CF Precision@10 for Fold 1: 0.0573
Weighted CF Recall@10 for Fold 1: 0.2386

--- Fold 2/5 ---
Weighted CF Precision@10 for Fold 2: 0.0574
Weighted CF Recall@10 for Fold 2: 0.2410

--- Fold 3/5 ---
Weighted CF Precision@10 for Fold 3: 0.0584
Weighted CF Recall@10 for Fold 3: 0.2478

--- Fold 4/5 ---
Weighted CF Precision@10 for Fold 4: 0.0566
Weighted CF Recall@10 for Fold 4: 0.2418

--- Fold 5/5 ---
Weighted CF Precision@10 for Fold 5: 0.0571
Weighted CF Recall@10 for Fold 5: 0.2403

--- Weighted CF Cross-Validation Results (Average over 5 folds) ---
Average Weighted CF Precision@10: 0.0574
Average Weighted CF Recall@10: 0.2419
